# 🎙️ COSSMIL — Estudio de voz (voz natural clonada de las `vof`)

Genera audios con **la voz real de la locutora de COSSMIL** — la de `assets/vof/`, que ya viene dentro de este cuaderno —: las **voces del Modo Guiado** y **cualquier texto** que escribas.

**Cómo suena natural:** en vez de una voz sintética "maquillada", un modelo de **clonación de voz** (Chatterbox Multilingual, código abierto, licencia MIT) escucha a la locutora y **habla como ella**: su timbre, su entonación y su ritmo. El cuaderno elige solo el fragmento de las `vof` que mejor se clona (celda 6), revisa cada frase y regenera las que salgan cortadas, pasa los números a palabras e iguala el volumen al de las `vof`.

### Uso
1. `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`.
2. `Entorno de ejecución → Ejecutar todo` y acepta el permiso de Google Drive.
3. La 1ª vez tarda ≈ 15 min (instala y elige la referencia); luego ≈ 5 min.
4. Se descarga **`vof_tutorial_….zip`** con las voces del Modo Guiado (y del tutorial): extrae los mp3 **directo** en `assets/vof_tutorial/`.
5. Para cualquier otro texto: celda **8 · Estudio**.

> **Calidad por frase:** se generan varias tomas; **Whisper** comprueba que se entienda completa (sin cortes ni balbuceos), **UTMOS** elige la más natural (la menos robótica) y **se recortan los sonidos extraños después de la última palabra** (solo si Whisper confirma que no son parte del texto); **Resemble Enhance** la limpia con red neuronal (a fuerza máxima) y la deja nítida a 44,1 kHz; además se silencian los huecos entre palabras, se limpia la referencia antes de clonar y el volumen se iguala con ganancia fija (sin subir el ruido de las pausas). Al final se listan las frases a revisar; rehazlas con otra **SEMILLA** (celda 7 → `SOLO_ESTOS`; celda 8 para textos libres). Más grabaciones limpias de la locutora en `MyDrive/cossmil_rvc/audio_extra/` también ayudan.

In [ ]:
#@title 1 · Configuración general
USAR_DRIVE = True        #@param {type:"boolean"}
#@markdown Guarda la referencia elegida, los audios extra y tus resultados en `MyDrive/cossmil_rvc`.
SUBIR_AUDIO_EXTRA = False  #@param {type:"boolean"}
#@markdown Pide subir grabaciones extra de la MISMA locutora (también puedes dejarlas en `MyDrive/cossmil_rvc/audio_extra/`).
REFORZAR_CON_RVC = False #@param {type:"boolean"}
#@markdown Opcional: además pasa la voz clonada por un modelo RVC entrenado con las vof (≈ +25 min la 1ª vez). Suele sonar **menos** natural; déjalo apagado salvo que quieras probarlo.
REENTRENAR = False       #@param {type:"boolean"}
EPOCHS = 300             #@param {type:"slider", min:100, max:800, step:50}
MODEL, SR = 'instructora', 40000

import os, subprocess
GPU = 'GPU' in subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print('GPU:', 'sí ✔' if GPU else 'NO — la voz clonada funciona en CPU pero MUY lento '
      '(Entorno de ejecución → Cambiar tipo → T4).')

BACKUP = None
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP = '/content/drive/MyDrive/cossmil_rvc'
    for d in ('modelo', 'audio_extra', 'salidas'):
        os.makedirs(f'{BACKUP}/{d}', exist_ok=True)
    print('Drive:', BACKUP)

In [ ]:
#@title 2 · Voz de la locutora (vof embebidas + audio extra)
import base64, glob, hashlib, os, shutil, subprocess
VOF = {"vof_01.mp3": "...", "vof_02.mp3": "...", "vof_03.mp3": "...", "vof_04.mp3": "...", "vof_05.mp3": "...", "vof_06.mp3": "...", "vof_07.mp3": "..."}
RAW, DATASET = '/content/raw_audio', '/content/rvc_dataset'
shutil.rmtree(RAW, ignore_errors=True); shutil.rmtree(DATASET, ignore_errors=True)
os.makedirs(RAW); os.makedirs(DATASET)
for name, b64 in VOF.items():
    open(f'{RAW}/{name}', 'wb').write(base64.b64decode(b64))

AUDIO_EXT = ('.mp3', '.wav', '.m4a', '.ogg', '.flac', '.aac', '.opus')
if BACKUP:
    for f in sorted(glob.glob(f'{BACKUP}/audio_extra/*')):
        if f.lower().endswith(AUDIO_EXT):
            shutil.copy(f, f'{RAW}/drive_{os.path.basename(f)}')
if SUBIR_AUDIO_EXTRA:
    from google.colab import files
    print('Sube grabaciones de la MISMA voz (sin música, sin eco, sin otras personas):')
    for name, data in files.upload().items():
        open(f'{RAW}/extra_{name}', 'wb').write(data)
        if BACKUP:  # para que la próxima vez no haya que subirlas de nuevo
            open(f'{BACKUP}/audio_extra/{name}', 'wb').write(data)

# Sin duplicados (el mismo audio subido dos veces) y huella del contenido: si agregas audio, se vuelve a elegir la referencia.
digests = {}
for f in sorted(glob.glob(f'{RAW}/*')):
    d = hashlib.sha256(open(f, 'rb').read()).digest()
    if d in digests: os.remove(f)
    else: digests[d] = f
HUELLA = hashlib.sha256(b''.join(sorted(digests))).hexdigest()[:16]

total = 0.0
for i, f in enumerate(sorted(glob.glob(f'{RAW}/*')), 1):
    out = f'{DATASET}/{os.path.splitext(os.path.basename(f))[0]}.wav'  # vof_03.wav, drive_x.wav…
    r = subprocess.run(['ffmpeg', '-y', '-i', f, '-ac', '1', '-ar', '44100', '-af',
                        'highpass=f=60,silenceremove=start_periods=1:start_threshold=-45dB:stop_periods=-1:stop_duration=0.6:stop_threshold=-45dB,loudnorm=I=-16:TP=-1.5:LRA=11',
                        out], capture_output=True, text=True)
    if r.returncode:
        print(f'  ⚠ se omite {os.path.basename(f)} (no es audio legible)'); continue
    d = float(subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                              '-of', 'default=nk=1:nw=1', out], capture_output=True, text=True).stdout or 0)
    total += d
    print(f'  {os.path.basename(f)[:40]:40s} {d:6.1f}s')
print(f'\nDataset: {len(glob.glob(DATASET + "/*.wav"))} clips, {total/60:.1f} min de voz · huella {HUELLA}')
if total < 90:
    print('Consejo: más grabaciones limpias de la locutora dan más referencias entre las que elegir.')

In [ ]:
#@title 3 · Instalar el motor de voz natural (Chatterbox, entorno aislado) — ~4 min la 1ª vez
import os
VENV = '/content/voz_env'
PY = f'{VENV}/bin/python'
if not os.path.exists(f'{VENV}/.ok'):
    !command -v uv > /dev/null || pip -q install uv
    !uv venv -q --python 3.11 {VENV}
    !uv pip install -q --python {PY} "chatterbox-tts==0.1.7" soundfile scipy
    !touch {VENV}/.ok
# perth (componente de chatterbox) importa pkg_resources: uv no trae setuptools en el entorno.
!uv pip install -q --python {PY} "setuptools<81" pyyaml noisereduce
!{PY} -c "import torch, chatterbox; from perth.perth_net.perth_net_implicit.perth_watermarker import PerthImplicitWatermarker; print('torch', torch.__version__, '· GPU' if torch.cuda.is_available() else '· CPU')"

# Nitidez de estudio: Resemble Enhance (MIT) limpia con red neuronal y reconstruye la voz a 44,1 kHz.
# Sus dependencias fijadas (torch 2.1, deepspeed) solo hacen falta para ENTRENARLO: se instala sin
# ellas y con un sustituto mínimo de deepspeed (que en inferencia nunca se llama).
if not os.path.exists(f'{VENV}/.ok_realce'):
    !command -v git-lfs > /dev/null || (apt-get -qq update > /dev/null && apt-get -qq install -y git-lfs > /dev/null)
    !git lfs install --skip-repo > /dev/null
    !uv pip install -q --python {PY} --no-deps resemble-enhance==0.0.1
    !uv pip install -q --python {PY} omegaconf rich resampy matplotlib pandas tqdm
    SITIO = !{PY} -c "import site; print(site.getsitepackages()[0])"
    DS = f'{SITIO[-1]}/deepspeed'
    if not os.path.exists(f'{DS}/runtime/engine.py'):
        for d in ('', '/accelerator', '/runtime'):
            os.makedirs(DS + d, exist_ok=True)
        _no = "raise RuntimeError('deepspeed no instalado: solo hace falta para entrenar Resemble Enhance')"
        open(f'{DS}/__init__.py', 'w').write(f"class DeepSpeedConfig:\n    def __init__(self, *a, **k): {_no}\n"
                                              f"def init_distributed(*a, **k): {_no}\n")
        open(f'{DS}/accelerator/__init__.py', 'w').write(f"def get_accelerator(*a, **k): {_no}\n")
        open(f'{DS}/runtime/__init__.py', 'w').write('')
        open(f'{DS}/runtime/engine.py', 'w').write('class DeepSpeedEngine:\n    pass\n')
        open(f'{DS}/runtime/utils.py', 'w').write(f"def clip_grad_norm_(*a, **k): {_no}\n")
    # descarga el modelo de realce ahora (git lfs, ~1 GB) para no hacerlo a mitad de una generación
    r = !{PY} -c "from resemble_enhance.enhancer.download import download; print('realce OK', download())" 2>&1
    print(r[-1])
    if 'realce OK' in r[-1]:
        !touch {VENV}/.ok_realce
    else:
        print('⚠ Nitidez no disponible (se generará igual, a 24 kHz). Detalle:', *r[-6:], sep='\n')
print('Motor de voz listo ✔ (entorno aparte: no toca el numpy/torch de Colab)')

In [ ]:
#@title 4 · (Opcional) Refuerzo de timbre con RVC — solo si marcaste REFORZAR_CON_RVC
VERSION_APPLIO = "probada (recomendada)"  #@param ["probada (recomendada)", "última (main)"]
PTH = INDEX = None
if REFORZAR_CON_RVC:
    import os
    APPLIO = '/content/Applio'
    if not os.path.exists(f'{APPLIO}/.cossmil_ok'):
        COMMIT = '939d9ede94d563eb5b96a55dc3922e03f93d4064' if VERSION_APPLIO.startswith('probada') else 'main'
        !rm -rf /content/Applio && git init -q /content/Applio
        %cd /content/Applio
        !git remote add origin https://github.com/IAHispano/Applio && git fetch -q --depth 1 origin {COMMIT} && git -c advice.detachedHead=false checkout -q FETCH_HEAD
        !apt-get -qq update > /dev/null && apt-get -qq install -y portaudio19-dev > /dev/null
        !command -v uv > /dev/null || pip -q install uv
        !uv pip install --system -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match
        !python core.py prerequisites --models --pretraineds-hifigan --exe
        !cp assets/config_template.json assets/config.json
        !touch .cossmil_ok
    %cd /content/Applio
    print('Applio listo ✔  (los avisos rojos de pip sobre "dependency conflicts" son inofensivos)')
    import glob, os, shutil, subprocess, time
    LOG = f'{APPLIO}/logs/{MODEL}'
    # PYTORCH_JIT=0: el torch de Colab (CUDA 13) no trae libnvrtc-builtins para compilar los kernels
    # TorchScript de RVC ("nvrtc: failed to open libnvrtc-builtins.so.13.0"); sin JIT corre igual.
    import site
    _libs = sorted({os.path.dirname(f) for d in site.getsitepackages()
                    for f in glob.glob(f'{d}/nvidia/**/libnvrtc*.so*', recursive=True)})
    ENV = {**os.environ, 'COLUMNS': '200', 'NO_COLOR': '1', 'TERM': 'dumb', 'PYTHONUNBUFFERED': '1',
           'PYTORCH_JIT': '0',
           'LD_LIBRARY_PATH': ':'.join(_libs + [os.environ.get('LD_LIBRARY_PATH', '')]).strip(':')}

    def applio(cmd, *args, mostrar=True):
        # Ejecuta `python core.py <cmd> ...` mostrando la salida en vivo (subprocess.run no la muestra en Colab).
        p = subprocess.Popen(['python', 'core.py', cmd, *map(str, args)], cwd=APPLIO, env=ENV, text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        salida = []
        for linea in p.stdout:
            salida.append(linea)
            if mostrar: print(linea, end='')
        texto = ''.join(salida)
        if p.wait() != 0:
            fallo(cmd, texto, mostrar)
        return texto

    def fallo(cmd, texto, mostrado=True):
        if not mostrado: print(texto[-3000:])
        raise RuntimeError(f'Falló "core.py {cmd}". Copia el error de arriba y pásamelo.')

    def exigir(cmd, texto, patron):
        # Applio suele terminar con código 0 aunque falle por dentro: se valida lo que dejó en disco.
        if not glob.glob(patron):
            fallo(cmd, texto)

    def modelo_en(d):
        pth = [p for p in glob.glob(f'{d}/*.pth') if not os.path.basename(p).startswith(('G_', 'D_'))]
        idx = glob.glob(f'{d}/*.index')
        pth.sort(key=os.path.getmtime); idx.sort(key=os.path.getmtime)
        return (pth[-1], idx[-1]) if pth and idx else (None, None)

    PTH, INDEX = modelo_en(f'{BACKUP}/modelo') if BACKUP else (None, None)
    huella_guardada = open(f'{BACKUP}/modelo/huella.txt').read().strip() if BACKUP and os.path.exists(f'{BACKUP}/modelo/huella.txt') else None
    entrenar = REENTRENAR or not PTH
    if PTH and not REENTRENAR:
        if huella_guardada == HUELLA:
            print('Modelo de Drive al día con el dataset → no se reentrena ✔')
        elif huella_guardada is None:
            print('Modelo de Drive de una versión anterior (sin huella): se reutiliza.\n'
                  'Si le agregaste audio desde entonces, marca REENTRENAR en la celda 1.')
        elif huella_guardada != HUELLA:
            print('El dataset cambió desde el último entrenamiento (¿audio extra nuevo?) → se reentrena.')
            entrenar = True

    if entrenar:
        assert GPU, 'Entrenar necesita GPU: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU, y Ejecutar todo.'
        t0 = time.time()
        shutil.rmtree(LOG, ignore_errors=True)
        ncpu = os.cpu_count() or 2
        t = applio('preprocess', '--model-name', MODEL, '--dataset-path', DATASET, '--sample-rate', SR,
                   '--cpu-cores', ncpu, '--cut-preprocess', 'Automatic', '--chunk-len', '3.0', '--overlap-len', '0.3')
        exigir('preprocess', t, f'{LOG}/sliced_audios/*.wav')
        t = applio('extract', '--model-name', MODEL, '--sample-rate', SR, '--f0-method', 'rmvpe',
                   '--cpu-cores', ncpu, '--gpu', '0', '--embedder-model', 'contentvec', '--include-mutes', 2)
        exigir('extract', t, f'{LOG}/extracted/*')
        t = applio('index', '--model-name', MODEL, '--index-algorithm', 'Auto')
        exigir('index', t, f'{LOG}/*.index')
        # Con poca voz, el lote por defecto (8) deja <3 lotes y Applio aborta "Not enough data": se ajusta solo.
        trozos = len(glob.glob(f'{LOG}/sliced_audios/*.wav'))
        lote = max(2, min(8, trozos // 6))
        print(f'\n{trozos} trozos de audio → batch_size {lote}, {EPOCHS} epochs')
        t = applio('train', '--model-name', MODEL, '--sample-rate', SR, '--total-epoch', EPOCHS,
               '--save-every-epoch', 50, '--save-only-latest', '--batch-size', lote, '--gpu', '0',
               '--vocoder', 'HiFi-GAN', '--pretrained')
        pths = sorted(glob.glob(f'{LOG}/{MODEL}_*e_*s.pth'), key=os.path.getmtime)
        idxs = sorted(glob.glob(f'{LOG}/*.index'), key=os.path.getmtime)
        if not pths or not idxs:
            fallo('train', t)
        PTH, INDEX = pths[-1], idxs[-1]
        if BACKUP:
            viejo = f'{BACKUP}/modelo_anterior'
            shutil.rmtree(viejo, ignore_errors=True); os.makedirs(viejo)
            for f in glob.glob(f'{BACKUP}/modelo/*'):
                shutil.move(f, viejo)
            for f in (PTH, INDEX):
                shutil.copy(f, f'{BACKUP}/modelo/')
            open(f'{BACKUP}/modelo/huella.txt', 'w').write(HUELLA)
            PTH, INDEX = modelo_en(f'{BACKUP}/modelo')
            print('Modelo respaldado en Drive (el anterior quedó en modelo_anterior/)')
        print(f'Entrenado en {(time.time() - t0) / 60:.0f} min')
    print('modelo:', PTH, '\nindex :', INDEX)
else:
    print('Omitido ✔ — se usa la voz clonada tal cual (lo más natural).')

In [ ]:
%%writefile /content/estudio_voz_lib.py
#@title 5 · Motor del estudio (no hace falta tocar nada aquí)
"""Motor del Estudio de voz COSSMIL (texto libre → voz de la locutora vof).

Este archivo se EMBEBE en la celda "Motor" del notebook de Colab al correr
`build_colab_notebook.py`. La parte de texto (parse_textos, segmentar, …) es
Python puro y se prueba localmente con `test_estudio_voz_lib.py`; la parte de
audio (clonación Chatterbox desde las vof → RVC opcional → ffmpeg) solo corre en Colab.

Formato del texto (una línea = un audio):

    # comentario (se ignora)
    bienvenida | Bienvenido a COSSMIL. [pausa] Le ayudaré con su cita.
    Esta línea no tiene nombre: se llamará clip_02_esta-linea-no-tiene.

Marcas dentro del texto: [pausa] (0,6 s), [pausa 1.5] o [pausa 800ms].
"""
import csv
import io
import json
import re
import unicodedata

PAUSA_DEFECTO = 0.6      # segundos de [pausa]
PAUSA_ENTRE_TROZOS = 0.25  # silencio al unir frases de un texto largo
MAX_CARACTERES = 400     # trozo máximo que se sintetiza de una vez

# Palabras que el TTS lee mal. Se aplican con límite de palabra y respetando mayúsculas.
PRONUNCIACION_DEFECTO = {
    'COSSMIL': 'Cossmil',
    'Dra.': 'doctora',
    'Dr.': 'doctor',
    'Nro.': 'número',
    'N°': 'número',
    'Sr.': 'señor',
    'Sra.': 'señora',
    'Cap.': 'capitán',
    'Tte.': 'teniente',
    'Cnl.': 'coronel',
}

_ID_VALIDO = re.compile(r'^[A-Za-z0-9][A-Za-z0-9_.-]{0,59}$')
_PAUSA = re.compile(r'\[\s*pausa(?:\s*[=:]?\s*(\d+(?:[.,]\d+)?)\s*(ms|s)?)?\s*\]', re.IGNORECASE)
_FIN_FRASE = re.compile(r'(?<=[.!?…;])\s+')


def slug(texto, palabras=4, largo=32):
    """'¡Hola, señor Pérez!' → 'hola-senor-perez' (seguro para nombre de archivo)."""
    t = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode().lower()
    t = re.sub(r'\[[^\]]*\]', ' ', t)
    partes = re.findall(r'[a-z0-9]+', t)[:palabras]
    return '-'.join(partes)[:largo].strip('-') or 'audio'


def nombre_seguro(nombre):
    """Limpia un id escrito por el usuario para usarlo como nombre de archivo."""
    base = unicodedata.normalize('NFKD', nombre).encode('ascii', 'ignore').decode()
    base = re.sub(r'\.(mp3|wav|ogg|m4a)$', '', base.strip(), flags=re.IGNORECASE)
    base = re.sub(r'[^A-Za-z0-9_.-]+', '_', base).strip('._-')
    return base[:60] or 'audio'


def _auto(contador, texto):
    contador[0] += 1
    return f'clip_{contador[0]:02d}_{slug(texto)}'


def _unicos(pares):
    vistos, salida = {}, []
    for nombre, texto in pares:
        n = nombre
        if n in vistos:
            vistos[n] += 1
            n = f'{nombre}_{vistos[nombre]}'
        else:
            vistos[n] = 1
        salida.append((n, texto))
    return salida


def parse_textos(crudo):
    """Texto del formulario → [(nombre, texto)]. Ver formato en el docstring del módulo."""
    pares, auto = [], [0]
    for linea in crudo.splitlines():
        linea = linea.strip()
        if not linea or linea.startswith('#'):
            continue
        nombre, texto = None, linea
        if '|' in linea:
            izq, der = linea.split('|', 1)
            if _ID_VALIDO.match(izq.strip()) and der.strip():
                nombre, texto = nombre_seguro(izq), der.strip()
        if not re.sub(_PAUSA, '', texto).strip():
            continue
        if nombre is None:
            nombre = _auto(auto, texto)
        pares.append((nombre, texto))
    return _unicos(pares)


def parse_archivo(nombre_archivo, datos):
    """Archivo subido (.txt / .json / .csv) → [(nombre, texto)]."""
    texto = datos.decode('utf-8-sig') if isinstance(datos, bytes) else datos
    ext = nombre_archivo.lower().rsplit('.', 1)[-1]
    auto = [0]
    if ext == 'json':
        obj = json.loads(texto)
        if isinstance(obj, dict) and isinstance(obj.get('lines'), (dict, list)):
            obj = obj['lines']
        if isinstance(obj, dict):
            pares = [(nombre_seguro(k), str(v).strip()) for k, v in obj.items()]
        else:
            pares = []
            for it in obj:
                if isinstance(it, str):
                    pares.append((_auto(auto, it), it.strip()))
                else:
                    t = str(it.get('text') or it.get('texto') or '').strip()
                    n = it.get('id') or it.get('nombre')
                    pares.append((nombre_seguro(str(n)) if n else _auto(auto, t), t))
        return _unicos([(n, t) for n, t in pares if t])
    if ext == 'csv':
        filas = [f for f in csv.reader(io.StringIO(texto)) if any(c.strip() for c in f)]
        if filas and [c.strip().lower() for c in filas[0][:2]] in (['id', 'text'], ['id', 'texto'], ['nombre', 'texto']):
            filas = filas[1:]
        pares = []
        for f in filas:
            if len(f) >= 2 and f[1].strip():
                pares.append((nombre_seguro(f[0]) if f[0].strip() else _auto(auto, f[1]), f[1].strip()))
            elif f[0].strip():
                pares.append((_auto(auto, f[0]), f[0].strip()))
        return _unicos(pares)
    return parse_textos(texto)


def aplicar_pronunciacion(texto, diccionario):
    """Reemplaza palabras mal leídas por el TTS (claves más largas primero)."""
    for clave in sorted(diccionario, key=len, reverse=True):
        izq = r'(?<!\w)' if clave[0].isalnum() else ''
        der = r'(?!\w)' if clave[-1].isalnum() else ''
        texto = re.sub(izq + re.escape(clave) + der, diccionario[clave], texto)
    return texto


_UNIDADES = ('cero uno dos tres cuatro cinco seis siete ocho nueve diez once doce trece catorce quince '
             'dieciséis diecisiete dieciocho diecinueve veinte veintiuno veintidós veintitrés veinticuatro '
             'veinticinco veintiséis veintisiete veintiocho veintinueve').split()
_DECENAS = {3: 'treinta', 4: 'cuarenta', 5: 'cincuenta', 6: 'sesenta', 7: 'setenta', 8: 'ochenta', 9: 'noventa'}
_CENTENAS = {1: 'ciento', 2: 'doscientos', 3: 'trescientos', 4: 'cuatrocientos', 5: 'quinientos',
             6: 'seiscientos', 7: 'setecientos', 8: 'ochocientos', 9: 'novecientos'}


def _apocope(palabras):
    """'veintiuno' → 'veintiún', 'treinta y uno' → 'treinta y un' (delante de un sustantivo o de mil)."""
    if palabras.endswith('veintiuno'):
        return palabras[:-9] + 'veintiún'
    return palabras[:-3] + 'un' if palabras.endswith('uno') else palabras


def _cardinal(n):
    """Entero (0 ≤ n < 10**12) → palabras en español ('ciento veintiuno')."""
    if n < 30:
        return _UNIDADES[n]
    if n < 100:
        d, u = divmod(n, 10)
        return _DECENAS[d] + (f' y {_UNIDADES[u]}' if u else '')
    if n < 1000:
        c, r = divmod(n, 100)
        if n == 100:
            return 'cien'
        return _CENTENAS[c] + (f' {_cardinal(r)}' if r else '')
    if n < 10**6:
        m, r = divmod(n, 1000)
        miles = 'mil' if m == 1 else f'{_apocope(_cardinal(m))} mil'
        return miles + (f' {_cardinal(r)}' if r else '')
    mm, r = divmod(n, 10**6)
    millones = 'un millón' if mm == 1 else f'{_apocope(_cardinal(mm))} millones'
    return millones + (f' {_cardinal(r)}' if r else '')


def numeros_a_palabras(texto):
    """Horas (8:30), porcentajes y enteros → palabras: el modelo de voz lee mal los dígitos.
    Deja intactos los números largos tipo código (≥ 7 dígitos) salvo que tengan separador de miles,
    y las marcas entre corchetes ([pausa 1.5])."""
    partes = re.split(r'(\[[^\]]*\])', texto)
    return ''.join(p if p.startswith('[') else _numeros(p) for p in partes)


def _numeros(texto):
    def hora(m):
        h, mi = int(m.group(1)), int(m.group(2))
        if h > 24 or mi > 59:
            return m.group(0)
        h_txt = 'una' if h == 1 else _cardinal(h)
        return h_txt if mi == 0 else f'{h_txt} y {_cardinal(mi)}'
    texto = re.sub(r'\b(\d{1,2}):(\d{2})\b', hora, texto)
    texto = re.sub(r'\b(\d{1,3}(?:\.\d{3})+)\b', lambda m: m.group(1).replace('.', ''), texto)
    texto = re.sub(r'(\d+)\s?%', lambda m: m.group(1) + ' por ciento', texto)

    def entero(m):
        dig, sigue = m.group(1), m.group(2) or ''
        if len(dig) >= 7:
            return m.group(0)
        palabras = _cardinal(int(dig))
        if sigue:  # apócope ante sustantivo: un médico, veintiún días
            palabras = _apocope(palabras)
        return palabras + sigue
    return re.sub(r'(?<![\w.,])(\d+)(?![\w]|[.,]\d)(\s+(?=[^\W\d_]))?', entero, texto)


def cerrar_frase(texto):
    """El modelo tiende a cortar o balbucear si la frase no termina en puntuación."""
    texto = texto.strip()
    return texto if not texto or texto[-1] in '.!?…' else texto.rstrip(',;:') + '.'


def _palabras(texto):
    t = unicodedata.normalize('NFKD', texto.lower()).encode('ascii', 'ignore').decode()
    return re.findall(r'[a-z0-9]+', t)


def coincidencia(esperado, oido):
    """Compara el texto pedido con lo que se entiende en el audio (transcripción).
    Devuelve (parecido 0–1 por palabras, ¿se oye el final?) — detecta cortes y balbuceos."""
    import difflib
    a, b = _palabras(numeros_a_palabras(esperado)), _palabras(numeros_a_palabras(oido))
    if not a:
        return 1.0, True
    if not b:
        return 0.0, False
    conocidas = set(a)
    for i, w in enumerate(b):  # variantes de escritura (Cossmil/Cosmil) cuentan como la misma palabra
        if w not in conocidas:
            cerca = max(conocidas, key=lambda c: difflib.SequenceMatcher(None, c, w).ratio())
            if difflib.SequenceMatcher(None, cerca, w).ratio() >= 0.8:
                b[i] = cerca
    parecido = difflib.SequenceMatcher(None, a, b, autojunk=False).ratio()
    cola = b[-4:]
    fin = any(difflib.SequenceMatcher(None, a[-1], w).ratio() >= 0.75 for w in cola)
    if len(a) >= 2:  # la penúltima también debe estar cerca del final
        fin = fin and any(difflib.SequenceMatcher(None, a[-2], w).ratio() >= 0.75 for w in b[-6:])
    return round(parecido, 3), fin


def puntuar_toma(parecido, fin, razon_duracion, snr_db=None, mos=None):
    """Nota de una toma: manda que se entienda completa; luego naturalidad (MOS 1–5, lo que
    separa una toma humana de una robótica), duración plausible y limpieza."""
    import math
    nota = parecido + (0.1 if fin else -0.35)
    nota -= 0.3 * max(0.0, abs(math.log(max(razon_duracion, 1e-3))) - math.log(1.5))
    if snr_db is not None:
        nota -= 0.01 * max(0.0, 40.0 - snr_db)
    if mos is not None:
        nota += 0.3 * (mos - 3.5)
    return round(nota, 4)


def silabas_estimadas(texto):
    """Sílabas aproximadas (grupos vocálicos) para prever cuánto debería durar un audio."""
    return max(1, len(re.findall(r'[aeiouáéíóúü]+', texto.lower())))


def _segundos(num, unidad):
    if num is None:
        return PAUSA_DEFECTO
    v = float(num.replace(',', '.'))
    return v / 1000 if (unidad or '').lower() == 'ms' else v


def _trocear(frase, max_car):
    """Parte un texto largo en trozos ≤ max_car cortando en fin de frase (o en comas)."""
    if len(frase) <= max_car:
        return [frase]
    trozos, actual = [], ''
    for oracion in _FIN_FRASE.split(frase):
        piezas = [oracion] if len(oracion) <= max_car else re.split(r'(?<=,)\s+', oracion)
        for p in piezas:
            while len(p) > max_car:  # sin puntuación: corte duro en el último espacio
                corte = p.rfind(' ', 0, max_car)
                corte = corte if corte > 0 else max_car
                if actual:
                    trozos.append(actual); actual = ''
                trozos.append(p[:corte].strip()); p = p[corte:].strip()
            if actual and len(actual) + 1 + len(p) > max_car:
                trozos.append(actual); actual = p
            else:
                actual = f'{actual} {p}'.strip()
    if actual:
        trozos.append(actual)
    return [t for t in trozos if t]


def segmentar(texto, max_car=MAX_CARACTERES, pausa_trozos=PAUSA_ENTRE_TROZOS):
    """Texto → [('habla', str) | ('silencio', segundos)] listo para sintetizar."""
    salida, pos = [], 0

    def agregar_habla(fragmento):
        fragmento = re.sub(r'\s+', ' ', fragmento).strip()
        if not fragmento:
            return
        for i, t in enumerate(_trocear(fragmento, max_car)):
            if i:
                salida.append(('silencio', pausa_trozos))
            salida.append(('habla', t))

    for m in _PAUSA.finditer(texto):
        agregar_habla(texto[pos:m.start()])
        salida.append(('silencio', _segundos(m.group(1), m.group(2))))
        pos = m.end()
    agregar_habla(texto[pos:])
    while salida and salida[0][0] == 'silencio':
        salida.pop(0)
    while salida and salida[-1][0] == 'silencio':
        salida.pop()
    return salida


# ─────────────────────────── Audio (solo Colab) ───────────────────────────

def _ffmpeg(*args):
    import subprocess
    r = subprocess.run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', *map(str, args)],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f'ffmpeg falló: {r.stderr[-800:]}')


def duracion(ruta):
    import subprocess
    out = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                          '-of', 'default=nk=1:nw=1', ruta], capture_output=True, text=True).stdout
    return float(out or 0)


def ensamblar(partes, ruta_salida):
    """partes = [('wav', ruta) | ('silencio', seg)] → un solo WAV (misma frecuencia)."""
    import numpy as np
    import soundfile as sf
    sr, bloques = None, []
    for tipo, valor in partes:
        if tipo == 'wav':
            audio, sr_i = sf.read(valor, dtype='float32', always_2d=False)
            if audio.ndim > 1:
                audio = audio.mean(axis=1)
            if sr is None:
                sr = sr_i
            elif sr_i != sr:  # p. ej. un trozo sin realce (24 kHz) entre trozos realzados (44,1 kHz)
                import librosa
                audio = librosa.resample(audio, orig_sr=sr_i, target_sr=sr)
            bloques.append(audio)
        else:
            bloques.append(('silencio', valor))
    sr = sr or 44100
    final = [np.zeros(int(b[1] * sr), dtype='float32') if isinstance(b, tuple) else b for b in bloques]
    sf.write(ruta_salida, np.concatenate(final) if final else np.zeros(1, 'float32'), sr)


def masterizar(ruta_wav, ruta_salida, formato='mp3', normalizar=True, cola=0.25, lufs=-16.0):
    """Recorta silencios de borde SIN comerse finales de palabra (umbral bajo y 150 ms de
    margen), quita zumbidos graves, funde entrada y salida, e iguala el volumen a `lufs` (el
    de la locutora) con una GANANCIA FIJA + limitador. (El loudnorm de una pasada es dinámico:
    sube las partes bajas y con ellas el ruido de las pausas.)"""
    import os
    import tempfile
    borde = 'silenceremove=start_periods=1:start_threshold=-58dB:start_silence=0.15'
    recorte = ['highpass=f=70', borde, 'areverse', borde, 'afade=t=in:d=0.06', 'areverse',
               'afade=t=in:d=0.02']
    fd, tmp = tempfile.mkstemp(suffix='.wav')
    os.close(fd)
    try:
        _ffmpeg('-i', ruta_wav, '-af', ','.join(recorte), '-ac', 1, '-ar', 44100, '-c:a', 'pcm_f32le', tmp)
        filtros = []
        if normalizar:
            objetivo = max(-30.0, min(-9.0, lufs))
            ganancia = max(-20.0, min(30.0, objetivo - medir_lufs(tmp)))
            filtros += [f'volume={ganancia:.2f}dB', 'alimiter=limit=0.89:attack=5:release=50:level=false']
        filtros.append(f'apad=pad_dur={cola}')
        args = ['-i', tmp, '-af', ','.join(filtros), '-ac', 1, '-ar', 44100]
        args += ['-b:a', '160k'] if formato == 'mp3' else ['-c:a', 'pcm_s16le']
        _ffmpeg(*args, ruta_salida)
    finally:
        os.remove(tmp)


def generar(pares, voz, carpeta, convertir, velocidad=0, tono_hz=0, pronunciacion=None,
            formato='mp3', normalizar=True, sintetizar=None, log=print, lufs=-16.0,
            sintetizar_lote=None, max_car=MAX_CARACTERES):
    """[(nombre, texto)] → {nombre: ruta_final}.

    Voz: `sintetizar_lote([[texto, ruta_wav], …])` genera todos los trozos de una vez
    (clonación: el modelo se carga una sola vez) o, si no se da, `sintetizar` trozo a trozo.
    `convertir(carpeta_entrada, carpeta_salida)` aplica RVC en lote y deja `<base>.wav`;
    con `convertir=None` se usa la voz tal cual.
    """
    import glob, os, shutil
    pronunciacion = PRONUNCIACION_DEFECTO if pronunciacion is None else pronunciacion
    base_dir, rvc_dir, fin_dir = (os.path.join(carpeta, d) for d in ('base', 'rvc', 'final'))
    for d in (base_dir, rvc_dir, fin_dir):
        shutil.rmtree(d, ignore_errors=True)
        os.makedirs(d)

    planes, trabajos = {}, []
    for i, (nombre, texto) in enumerate(pares, 1):
        plan = []
        limpio = numeros_a_palabras(aplicar_pronunciacion(texto, pronunciacion))
        for j, (tipo, valor) in enumerate(segmentar(limpio, max_car=max_car)):
            if tipo == 'habla':
                valor = cerrar_frase(valor)
                pieza = f'{i:03d}_{j:03d}'
                trabajos.append([valor, os.path.join(base_dir, pieza + '.wav')])
                plan.append(('wav', pieza))
            else:
                plan.append(('silencio', valor))
        planes[nombre] = plan

    if sintetizar_lote:
        log(f'  generando {len(trabajos)} frase(s) con la voz clonada…')
        sintetizar_lote(trabajos)
    else:
        for n, (valor, ruta) in enumerate(trabajos, 1):
            sintetizar(valor, ruta, voz, velocidad, tono_hz)
            log(f'  [{n}/{len(trabajos)}] texto base')

    if convertir:
        log('  reforzando el timbre con RVC…')
        convertir(base_dir, rvc_dir)
    else:
        for f in glob.glob(os.path.join(base_dir, '*.wav')):
            shutil.copy(f, rvc_dir)

    salidas = {}
    for nombre, plan in planes.items():
        partes = []
        for tipo, valor in plan:
            if tipo == 'wav':
                ruta = os.path.join(rvc_dir, valor + '.wav')
                if not os.path.exists(ruta):
                    raise RuntimeError(f'No se generó {valor}.wav (¿falló la voz o RVC?). '
                                       f'Archivos en salida: {sorted(os.listdir(rvc_dir))[:5]}')
                partes.append(('wav', ruta))
            else:
                partes.append(('silencio', valor))
        crudo = os.path.join(rvc_dir, f'_{nombre}.wav')
        ensamblar(partes, crudo)
        final = os.path.join(fin_dir, f'{nombre}.{formato}')
        masterizar(crudo, final, formato, normalizar, lufs=lufs)
        salidas[nombre] = final
    for f in glob.glob(os.path.join(rvc_dir, '_*.wav')):
        os.remove(f)
    return salidas


# ─────────────────────── Calibración contra la voz real ───────────────────────

def _cargar_16k(rutas):
    import librosa
    import numpy as np
    partes = [librosa.load(r, sr=16000, mono=True)[0] for r in rutas]
    return np.concatenate(partes) if partes else np.zeros(16000, 'float32')


def analizar_voz(rutas):
    """Rasgos objetivos de una voz: tono (mediana de F0 en Hz), rango tonal
    (semitonos p10–p90), ritmo (sílabas/s aprox. por picos de energía) y segundos de habla."""
    import librosa
    import numpy as np
    from scipy.signal import find_peaks
    y = _cargar_16k([rutas] if isinstance(rutas, str) else rutas)
    f0, sonoro, _ = librosa.pyin(y, fmin=90, fmax=450, sr=16000, frame_length=1024, hop_length=160)
    f0 = f0[sonoro & ~np.isnan(f0)]
    if len(f0) < 10:
        raise RuntimeError('No se detectó voz suficiente para analizar.')
    # Envolvente de energía de la banda vocal (100 cuadros/s) → cada pico ≈ una sílaba
    banda = librosa.effects.preemphasis(y)
    rms = librosa.feature.rms(y=banda, frame_length=400, hop_length=160)[0]
    rms = np.convolve(rms, np.hanning(7) / np.hanning(7).sum(), mode='same')
    umbral = 0.12 * np.percentile(rms, 95)
    habla = rms > umbral
    # Une huecos cortos (<250 ms: oclusivas, valles entre sílabas) → tiempo de habla, sin contar pausas.
    huecos = np.flatnonzero(np.diff(np.concatenate([[1], habla.astype(int), [1]])))
    for ini, fin in zip(huecos[::2], huecos[1::2]):
        if 0 < ini and fin < len(habla) and fin - ini < 25:
            habla[ini:fin] = True
    picos, _ = find_peaks(rms, height=umbral, distance=8, prominence=0.08 * np.percentile(rms, 95))
    seg_habla = max(habla.sum() / 100, 0.1)
    return {
        'f0': float(np.median(f0)),
        'rango_st': float(12 * np.log2(np.percentile(f0, 90) / np.percentile(f0, 10))),
        'silabas_s': float(len(picos) / seg_habla),
        'seg_habla': float(seg_habla),
    }


def distancia_rasgos(ref, otro):
    """Qué tan lejos está una salida de la locutora en tono, ritmo y expresividad (0 = igual)."""
    import math
    return (abs(12 * math.log2(otro['f0'] / ref['f0']))            # semitonos
            + 4 * abs(math.log(max(otro['silabas_s'], 0.1) / max(ref['silabas_s'], 0.1)))  # ritmo
            + 0.3 * abs(otro['rango_st'] - ref['rango_st']))       # entonación


def medir_lufs(ruta):
    """Sonoridad integrada (LUFS) con ffmpeg loudnorm."""
    import json as _json, re as _re, subprocess
    r = subprocess.run(['ffmpeg', '-hide_banner', '-nostats', '-i', ruta, '-af',
                        'loudnorm=print_format=json', '-f', 'null', '-'], capture_output=True, text=True)
    m = _re.search(r'\{[^{}]*"input_i"[^{}]*\}', r.stderr)
    return float(_json.loads(m.group(0))['input_i']) if m else -16.0


def analizar_varias(entradas):
    """analizar_voz para varias entradas (cada una: ruta o lista de rutas) cargando librosa una vez."""
    return [analizar_voz(e) for e in entradas]


def similitudes(referencias, candidatos):
    """Similitud de hablante (coseno, WavLM-SV) de cada candidato contra el promedio de las
    referencias. Devuelve {'sims': [...]} o {'sims': None, 'aviso': motivo} si no hay red."""
    try:
        import librosa
        import numpy as np
        import torch
        from transformers import AutoFeatureExtractor, WavLMForXVector
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        fe = AutoFeatureExtractor.from_pretrained('microsoft/wavlm-base-plus-sv')
        red = WavLMForXVector.from_pretrained('microsoft/wavlm-base-plus-sv').to(dev).eval()

        def vector(ruta):
            y = librosa.load(ruta, sr=16000, mono=True)[0][:16000 * 20]
            x = fe(y, sampling_rate=16000, return_tensors='pt').to(dev)
            with torch.no_grad():
                e = red(**x).embeddings[0]
            return torch.nn.functional.normalize(e, dim=-1).cpu().numpy()

        ref = np.mean([vector(r) for r in referencias], axis=0)
        ref = ref / np.linalg.norm(ref)
        return {'sims': [float(vector(c) @ ref) for c in candidatos]}
    except Exception as e:  # noqa: BLE001 — sin la red se calibra solo con tono/ritmo
        return {'sims': None, 'aviso': f'{type(e).__name__}: {str(e)[:160]}'}


def preparar_referencias(wavs, carpeta, seg_min=8.0):
    """Referencias para clonar: cada vof sin silencios de borde y con pausas internas acortadas.
    La candidata i empieza con la vof i (el modelo toma de ahí la prosodia, primeros 6-10 s) y
    sigue con las demás (la huella de la voz se calcula sobre todo el audio)."""
    import os
    import librosa
    import numpy as np
    import soundfile as sf
    os.makedirs(carpeta, exist_ok=True)
    limpios = []
    for w in wavs:
        y, sr = librosa.load(w, sr=24000, mono=True)
        trozos = [y[a:b] for a, b in librosa.effects.split(y, top_db=35)]
        pausa = np.zeros(int(0.25 * sr), 'float32')
        partes = []
        for t in trozos:
            partes += [t, pausa]
        limpios.append(np.concatenate(partes[:-1]) if partes else y)
    salidas = []
    for i, y in enumerate(limpios):
        resto = [z for j, z in enumerate(limpios) if j != i]
        todo = np.concatenate([y] + resto)
        ruta = os.path.join(carpeta, f'ref_{os.path.splitext(os.path.basename(wavs[i]))[0]}.wav')
        sf.write(ruta, (0.9 * todo / max(1e-6, np.abs(todo).max())).astype('float32'), 24000)
        salidas.append({'ruta': ruta, 'seg_propios': round(len(y) / 24000, 1)})
    return salidas


def _modelo_clonacion():
    import torch
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS
    return ChatterboxMultilingualTTS.from_pretrained(device='cuda' if torch.cuda.is_available() else 'cpu')


def _cargar_asr(nombre='openai/whisper-large-v3-turbo'):
    """Whisper para 'escuchar' cada toma. Devuelve transcribir(y, sr) o None si no se puede."""
    try:
        import librosa
        import torch
        from transformers import WhisperForConditionalGeneration, WhisperProcessor
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        tipo = torch.float16 if dev == 'cuda' else torch.float32
        proc = WhisperProcessor.from_pretrained(nombre)
        red = WhisperForConditionalGeneration.from_pretrained(nombre).to(dev, dtype=tipo).eval()

        def transcribir(y, sr):
            y16 = librosa.resample(y, orig_sr=sr, target_sr=16000) if sr != 16000 else y
            x = proc(y16, sampling_rate=16000, return_tensors='pt').input_features.to(dev, dtype=tipo)
            with torch.no_grad():
                ids = red.generate(x, language='es', task='transcribe', max_new_tokens=220)
            return proc.batch_decode(ids, skip_special_tokens=True)[0]
        return transcribir
    except Exception as e:  # noqa: BLE001
        print(f'  [aviso] verificación con Whisper no disponible ({type(e).__name__}: {str(e)[:120]}); '
              'se usa solo la duración', flush=True)
        return None


def _cargar_mos():
    """UTMOS (predictor de naturalidad, MOS 1–5). Devuelve mos(y, sr) o None."""
    try:
        import librosa
        import torch
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        red = torch.hub.load('tarepan/SpeechMOS:v1.2.0', 'utmos22_strong', trust_repo=True).to(dev).eval()

        def mos(y, sr):
            y16 = librosa.resample(y, orig_sr=sr, target_sr=16000) if sr != 16000 else y
            with torch.no_grad():
                return float(red(torch.from_numpy(y16).float().unsqueeze(0).to(dev), 16000).item())
        return mos
    except Exception as e:  # noqa: BLE001
        print(f'  [aviso] medidor de naturalidad (UTMOS) no disponible ({type(e).__name__}: {str(e)[:100]})', flush=True)
        return None


def _cargar_realce(fuerza=0.9):
    """Resemble Enhance (MIT): quita ruido con una red neuronal y reconstruye la voz a 44,1 kHz
    (más nítida que los 24 kHz del modelo de voz). Devuelve realzar(y, sr) -> (y, sr) o None."""
    try:
        import torch
        from resemble_enhance.enhancer.inference import enhance
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'

        def realzar(y, sr):
            wav, sr2 = enhance(torch.from_numpy(y).float(), sr, dev, nfe=64, solver='midpoint',
                               lambd=fuerza, tau=0.5)
            return wav.detach().cpu().numpy().astype('float32'), int(sr2)
        return realzar
    except Exception as e:  # noqa: BLE001
        print(f'  [aviso] realce de nitidez (Resemble Enhance) no disponible ({type(e).__name__}: {str(e)[:100]})',
              flush=True)
        return None


def relacion_senal_ruido(y, sr):
    """dB entre la voz (percentil 95 de energía) y el fondo (percentil 10)."""
    import numpy as np
    marco = int(0.02 * sr)
    n = len(y) // marco
    if n < 10:
        return 60.0
    e = np.sqrt(np.mean(y[:n * marco].reshape(n, marco) ** 2, axis=1)) + 1e-9
    return float(20 * np.log10(np.percentile(e, 95) / np.percentile(e, 10)))


_AVISOS = {}


def tramos_de_voz(y, sr, umbral_db=-35.0, hueco_max=0.15, minimo=0.05):
    """[(inicio, fin)] en muestras de los tramos con voz: cuadros de 10 ms sobre `umbral_db`
    respecto a la voz fuerte, uniendo huecos cortos (< `hueco_max` s) y sin tramos mínimos."""
    import numpy as np
    marco = max(1, int(0.01 * sr))
    n = len(y) // marco
    if n == 0:
        return []
    e = np.sqrt(np.mean(np.asarray(y[:n * marco], dtype='float64').reshape(n, marco) ** 2, axis=1)) + 1e-9
    voz = 20 * np.log10(e / np.percentile(e, 95)) > umbral_db
    tramos, i = [], 0
    while i < n:
        if voz[i]:
            j = i
            while j < n and voz[j]:
                j += 1
            tramos.append([i, j])
            i = j
        else:
            i += 1
    unidos = []
    for t in tramos:
        if unidos and (t[0] - unidos[-1][1]) * 0.01 < hueco_max:
            unidos[-1][1] = t[1]
        else:
            unidos.append(t)
    return [(a * marco, b * marco) for a, b in unidos if (b - a) * 0.01 >= minimo]


def recortar_bordes(y, sr, texto, asr=None, margen_ini=0.08, margen_fin=0.15, max_cortes=3):
    """Quita lo que sobra antes de la primera palabra y DESPUÉS de la última (el modelo a veces
    agrega al final un respiro, un murmullo o sonidos sueltos). Un tramo final se corta solo si,
    sin él, Whisper sigue oyendo el texto completo; sin Whisper, solo si es un chasquido corto
    (< 0,25 s) y separado de la voz (> 0,3 s). Termina con fundido suave.
    Devuelve (y, oido, parecido, fin, segundos_recortados)."""
    import numpy as np
    y = np.asarray(y, dtype='float32')
    largo = len(y)
    tramos = tramos_de_voz(y, sr)
    oido, parecido, fin = None, None, True
    if asr:
        oido = asr(y, sr)
        parecido, fin = coincidencia(texto, oido)
    if tramos:
        fin_voz = len(tramos)
        for _ in range(max_cortes):
            if fin_voz <= 1:
                break
            ultimo, previo = tramos[fin_voz - 1], tramos[fin_voz - 2]
            corte = min(largo, previo[1] + int(margen_fin * sr))
            if asr:
                oido_c = asr(y[:corte], sr)
                parecido_c, fin_c = coincidencia(texto, oido_c)
                if not (fin_c and parecido_c >= (parecido or 0) - 0.01):
                    break
                oido, parecido, fin = oido_c, parecido_c, fin_c
            else:
                corto = (ultimo[1] - ultimo[0]) / sr < 0.25
                separado = (ultimo[0] - previo[1]) / sr > 0.3
                if not (corto and separado):
                    break
            fin_voz -= 1
        ini = max(0, tramos[0][0] - int(margen_ini * sr))
        fin_m = min(largo, tramos[fin_voz - 1][1] + int(margen_fin * sr))
        y = y[ini:fin_m].copy()
        f_in, f_out = min(len(y), int(0.01 * sr)), min(len(y), int(0.06 * sr))
        if f_in:
            y[:f_in] *= np.linspace(0, 1, f_in, dtype='float32')
        if f_out:
            y[-f_out:] *= np.linspace(1, 0, f_out, dtype='float32')
    return y, oido, parecido, fin, round((largo - len(y)) / sr, 2)


def silenciar_pausas(y, sr, atenuacion_db=-35.0, sosten=0.08):
    """Baja SOLO los huecos entre palabras (donde se oye el soplido de fondo). El umbral se
    adapta a cada audio: 8 dB sobre su piso de ruido (percentil 10), entre −45 y −25 dB respecto
    a la voz fuerte. Lo que queda debajo se atenúa `atenuacion_db`, con 80 ms de margen alrededor
    de la voz (protege inicios y finales de palabra) y transiciones suaves."""
    import numpy as np
    y = np.asarray(y, dtype='float32')
    marco = max(1, int(0.01 * sr))
    n = len(y) // marco
    if n < 20:
        return y
    e = np.sqrt(np.mean(y[:n * marco].reshape(n, marco) ** 2, axis=1)) + 1e-9
    db = 20 * np.log10(e / np.percentile(e, 95))
    umbral_db = min(-25.0, max(float(np.percentile(db, 10)) + 8.0, -45.0))
    voz = db > umbral_db
    k = max(1, int(sosten / 0.01))
    voz = np.convolve(voz.astype(float), np.ones(2 * k + 1), mode='same') > 0  # margen a ambos lados
    g = np.where(voz, 1.0, 10 ** (atenuacion_db / 20))
    g = np.convolve(np.pad(g, 1, mode='edge'), np.ones(3) / 3, mode='valid')  # ~30 ms de transición
    ganancia = np.repeat(g, marco)
    ganancia = np.concatenate([ganancia, np.full(len(y) - len(ganancia), ganancia[-1])])
    return (y * ganancia).astype('float32')


def limpiar_ruido(y, sr, fuerza=0.9):
    """Quita zumbido grave y ruido de fondo (reducción espectral) sin tocar la voz."""
    try:
        import noisereduce as nr
        from scipy.signal import butter, sosfiltfilt
        y = sosfiltfilt(butter(4, 70, 'highpass', fs=sr, output='sos'), y).astype('float32')
        return nr.reduce_noise(y=y, sr=sr, stationary=True, prop_decrease=fuerza,
                               n_std_thresh_stationary=1.5).astype('float32')
    except Exception as e:  # noqa: BLE001
        if not _AVISOS.get('limpieza'):
            _AVISOS['limpieza'] = True
            print(f'  [aviso] limpieza de ruido omitida ({type(e).__name__}: {str(e)[:100]})', flush=True)
        return y


def _clonar(modelo, trabajos, ajustes, mostrar=True, asr=None, mos=None, realzar=None):
    """Lee cada [texto, ruta_wav] con la voz ya condicionada en `modelo`.

    Por frase genera varias tomas (al menos `tomas_min`, hasta `intentos`): cada una se
    'escucha' con Whisper (¿se entiende completa?) y se puntúa su naturalidad con UTMOS. Se
    queda la mejor y solo esa pasa por el realce de nitidez (Resemble Enhance, 44,1 kHz)."""
    import os
    import random
    import numpy as np
    import soundfile as sf
    import torch
    ritmo = ajustes.get('silabas_s', 5.5)
    max_tomas = max(1, int(ajustes.get('intentos', 5)))
    min_tomas = min(max_tomas, max(1, int(ajustes.get('tomas_min', 3))))
    informe = []
    for n, (texto, ruta) in enumerate(trabajos):
        esperado = silabas_estimadas(texto) / ritmo
        mejor = None
        for toma in range(max_tomas):
            semilla = int(ajustes.get('semilla', 1234)) + 1000 * toma + n
            random.seed(semilla); np.random.seed(semilla); torch.manual_seed(semilla)
            wav = modelo.generate(texto, language_id='es',
                                  exaggeration=ajustes.get('exageracion', 0.5),
                                  cfg_weight=ajustes.get('cfg', 0.4),
                                  temperature=ajustes.get('temperatura', 0.75))
            y = wav.squeeze(0).detach().cpu().numpy().astype('float32')
            y, oido, parecido, fin, recorte = recortar_bordes(y, modelo.sr, texto, asr)
            razon = (len(y) / modelo.sr) / max(esperado, 0.3)
            snr = relacion_senal_ruido(y, modelo.sr)
            natural = mos(y, modelo.sr) if mos else None
            nota = puntuar_toma(parecido if parecido is not None else 1.0, fin, razon, snr, natural)
            completa = 0.6 <= razon <= 1.7 and (parecido is None or (parecido >= 0.92 and fin))
            if mejor is None or nota > mejor['nota']:
                mejor = dict(y=y, nota=nota, razon=razon, parecido=parecido, fin=fin, oido=oido,
                             snr=snr, mos=natural, completa=completa, recorte=recorte)
            if toma + 1 >= min_tomas and mejor['completa']:
                break
        y, sr = mejor['y'], modelo.sr
        if realzar:
            try:
                y, sr = realzar(y, sr)
            except Exception as e:  # noqa: BLE001
                print(f'  [aviso] realce omitido en esta frase ({type(e).__name__})', flush=True)
        elif ajustes.get('limpiar_ruido', False) and mejor['snr'] < 30:
            y = limpiar_ruido(y, sr, 0.6)  # respaldo suave solo si hay ruido de verdad
        if ajustes.get('silenciar_pausas', True):
            y = silenciar_pausas(y, sr)
        y = recortar_bordes(y, sr, texto, None, max_cortes=0)[0]  # cierre limpio tras el realce
        sf.write(ruta, y, sr)
        ok = mejor['parecido'] is None or (mejor['parecido'] >= 0.85 and mejor['fin'])
        informe.append({'ruta': os.path.basename(ruta), 'texto': texto, 'oido': mejor['oido'],
                        'parecido': mejor['parecido'], 'fin': mejor['fin'], 'razon': round(mejor['razon'], 2),
                        'snr': round(mejor['snr'], 1), 'mos': None if mejor['mos'] is None else round(mejor['mos'], 2),
                        'tomas': toma + 1, 'ok': ok, 'cola_recortada_s': mejor['recorte']})
        if mostrar:
            nat = f" · naturalidad {mejor['mos']:.2f}/5" if mejor['mos'] is not None else ''
            marca = '' if ok else f" ⚠ revisar (se oyó: «{(mejor['oido'] or '')[:70]}»)"
            print(f"  [{n + 1}/{len(trabajos)}] {texto[:60]} · {toma + 1} tomas{nat}{marca}", flush=True)
    return informe


def limpiar_referencia(ruta):
    """Pasa la referencia por el limpiador neuronal de Resemble Enhance (solo quita ruido, no
    cambia la voz) y la guarda al lado como *_limpia.wav. Si no se puede, usa la original."""
    import os
    destino = ruta[:-4] + '_limpia.wav'
    if os.path.exists(destino) and os.path.getmtime(destino) >= os.path.getmtime(ruta):
        return destino
    try:
        import librosa
        import soundfile as sf
        import torch
        from resemble_enhance.enhancer.inference import denoise
        y, sr = librosa.load(ruta, sr=44100, mono=True)
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        limpio, sr2 = denoise(torch.from_numpy(y).float(), sr, dev)
        sf.write(destino, limpio.detach().cpu().numpy(), int(sr2))
        print('  referencia de la voz limpiada ✔', flush=True)
        return destino
    except Exception as e:  # noqa: BLE001
        print(f'  [aviso] no se pudo limpiar la referencia ({type(e).__name__}); se usa la original', flush=True)
        return ruta


def clonar_lote(trabajos, referencia, ajustes):
    """Chatterbox Multilingual (MIT): clona la voz de `referencia` y lee cada [texto, ruta_wav]."""
    modelo = _modelo_clonacion()
    if ajustes.get('limpiar_referencia', True):
        referencia = limpiar_referencia(referencia)
    modelo.prepare_conditionals(referencia, exaggeration=ajustes.get('exageracion', 0.5))
    asr = _cargar_asr(ajustes.get('asr', 'openai/whisper-large-v3-turbo')) if ajustes.get('verificar', True) else None
    mos = _cargar_mos() if ajustes.get('naturalidad', True) else None
    realzar = _cargar_realce(ajustes.get('fuerza_realce', 0.9)) if ajustes.get('nitidez', True) else None
    return _clonar(modelo, trabajos, ajustes, asr=asr, mos=mos, realzar=realzar)


def clonar_candidatas(frase, referencias, carpeta, ajustes):
    """Lee la misma frase con cada referencia candidata (modelo cargado una sola vez)."""
    import os
    os.makedirs(carpeta, exist_ok=True)
    modelo = _modelo_clonacion()
    salidas = []
    for i, ref in enumerate(referencias):
        modelo.prepare_conditionals(ref, exaggeration=ajustes.get('exageracion', 0.5))
        ruta = os.path.join(carpeta, f'cand_{i + 1:02d}.wav')
        _clonar(modelo, [[frase, ruta]], dict(ajustes, intentos=1, tomas_min=1), mostrar=False)
        print(f'  referencia {i + 1}/{len(referencias)} lista', flush=True)
        salidas.append(ruta)
    return salidas


if __name__ == '__main__':
    # En Colab lo que usa numpy/librosa/torch corre en un proceso aparte, dentro del entorno
    # aislado de Chatterbox: así nunca choca con el numpy del kernel ni con el torch de Applio.
    import sys
    _resultado = globals()[sys.argv[1]](*json.loads(sys.argv[2]))
    print('@@RESULTADO@@' + json.dumps(_resultado))


In [ ]:
#@title 5b · Enlace del motor (no hace falta tocar nada aquí)
# Todo lo que usa numpy/librosa/torch corre en un proceso aparte con el Python del entorno de voz.
import datetime, glob, importlib, json, os, shutil, subprocess, sys
sys.path.insert(0, '/content')
import estudio_voz_lib as L
importlib.reload(L)
from estudio_voz_lib import (PRONUNCIACION_DEFECTO, parse_textos, parse_archivo, nombre_seguro,
                             duracion, distancia_rasgos, medir_lufs)
from IPython.display import Audio, HTML, display

def remoto(funcion, *args, mostrar=False):
    p = subprocess.Popen([PY, '/content/estudio_voz_lib.py', funcion, json.dumps(args)], cwd='/content',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, 'PYTORCH_JIT': '0', 'TOKENIZERS_PARALLELISM': 'false',
                              # checkpoint oficial de Resemble Enhance (formato deepspeed): torch 2.6 lo
                              # rechazaría con su carga "solo pesos" por defecto
                              'TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD': '1'})
    salida, resultado, listo = [], None, False
    for linea in p.stdout:
        if linea.startswith('@@RESULTADO@@'):
            resultado, listo = json.loads(linea[len('@@RESULTADO@@'):]), True
        else:
            salida.append(linea)
            if mostrar and linea.startswith('  ['): print(linea, end='')
    if p.wait() != 0 or not listo:
        raise RuntimeError(f'"{funcion}" falló:\n' + ''.join(salida)[-2500:])
    return resultado

L.ensamblar = lambda partes, ruta: remoto('ensamblar', partes, ruta)
analizar_varias = lambda entradas: remoto('analizar_varias', entradas)
similitudes = lambda refs, cands: remoto('similitudes', refs, cands)

GUION = {
 "invite": "¡Bienvenido a COSSMIL! Soy tu instructora. Te puedo enseñar a sacar una ficha, es decir, una cita médica, paso a paso. Toma menos de un minuto, y puedes repetir el tutorial cuando quieras, desde tu Perfil.",
 "ficha_00": "¡Hola! Vamos a sacar tu primera ficha juntos. Todo empieza aquí, en Inicio. Toca la primera opción del menú, el botón verde, Nueva Reserva.",
 "ficha_01": "¡Muy bien! Así se inicia una reserva. Ahora, elige tu hospital o policlínico. Estos son los que tienes habilitados, agrupados por regional.",
 "ficha_02": "¡Muy bien! Ahora, elige la especialidad médica que necesitas.",
 "ficha_03": "Estos son los médicos disponibles para esa especialidad. Elige el que prefieras.",
 "ficha_04": "Ahora, elige el día. Cada tarjeta te muestra si el médico atiende, y si quedan fichas.",
 "ficha_05": "¡Ya casi terminamos! Elige un horario disponible, dentro del día que escogiste.",
 "ficha_06": "Revisa que todos los datos estén correctos. Luego, toca Confirmar Reserva. No te preocupes, aquí no se creará ninguna cita real.",
 "ficha_07": "¡Misión cumplida! Esto fue solo una demostración, y no se creó ninguna cita real. Puedes ver tu ficha de ejemplo, o volver al inicio.",
 "calendario_00": "¡Hola! Te voy a enseñar a consultar los horarios de los médicos. Empezamos desde Inicio. Toca la tarjeta Calendario de Atención.",
 "calendario_01": "Aquí puedes ver los días y horarios en que atiende cada médico, sin reservar nada. Empieza eligiendo tu hospital o policlínico.",
 "calendario_02": "¡Muy bien! Ahora, elige la especialidad que quieres consultar.",
 "calendario_03": "Estos son los médicos de esa especialidad. Toca uno, para ver su horario de atención.",
 "calendario_04": "¡Eso es todo! Aquí ves los días, turnos y horas en que atiende este médico. Recuerda que esto es solo una consulta. Para sacar una ficha, usa Nueva Reserva, en Inicio. Puedes repetir este tutorial desde tu Perfil.",
 "tramites_00": "¡Hola! Vamos a generar un trámite, paso a paso. Empezamos desde Inicio. Toca la tarjeta Procedimientos COSSMIL.",
 "tramites_01": "Aquí puedes generar documentos oficiales, con tus datos ya cargados. Los trámites se organizan por gerencia. Entra a Gerencia de Salud.",
 "tramites_02": "¡Muy bien! Esta gerencia agrupa sus dependencias. Entra a Hospital.",
 "tramites_03": "Ya casi llegamos. Cada categoría agrupa documentos. Abre Formularios.",
 "tramites_04": "¡Listo! Cada tarjeta abre un formulario con tu nombre y cédula ya completados. Puedes imprimirlo, compartirlo o descargarlo. Y puedes repetir este tutorial desde tu Perfil.",
 "guiado_intro": "Bienvenido a la reserva guiada de COSSMIL. Le acompañaré paso a paso. Tenga en cuenta que esta reserva es real, y su cita quedará registrada. Comencemos.",
 "guiado_regional": "Primero, elija el hospital o policlínico donde desea atenderse. Están ordenados por regional.",
 "guiado_especialidad": "Ahora, elija la especialidad médica que necesita.",
 "guiado_medico": "Muy bien. Elija al médico con quien desea atenderse.",
 "guiado_dia": "Elija el día de su cita. Cada tarjeta le muestra si el médico atiende, y si hay fichas disponibles.",
 "guiado_hora": "Ahora, elija el horario que prefiera.",
 "guiado_confirmar": "Revise que sus datos sean correctos. Cuando esté listo, presione Confirmar, y su cita quedará registrada.",
 "guiado_final": "Su cita fue registrada con éxito. Puede ver o descargar su ficha cuando lo necesite. Gracias por confiar en COSSMIL."
}
NOMBRES_VOF = {"vof_01": "AUDIO 1. BIENVENIDA", "vof_02": "AUDIO 2. LOGIN", "vof_03": "AUDIO 3. ADVERTENCIA DE SEGURIDAD", "vof_04": "AUDIO 4. HORARIOS HABILITADOS CON HORA", "vof_05": "AUDIO 4.1. HORARIO HABILITADO SIN HORA", "vof_06": "AUDIO 5. FINAL CITA MEDICA REGISTRADA", "vof_07": "AUDIO 7. NOTIFICACION CITA MEDICA"}
AJUSTES = dict(exageracion=0.5, cfg=0.4, temperatura=0.75, semilla=1234, intentos=5, tomas_min=3, silabas_s=5.5,
               verificar=True, asr='openai/whisper-large-v3-turbo', naturalidad=True,
               nitidez=True, fuerza_realce=0.9, limpiar_ruido=False, silenciar_pausas=True,
               limpiar_referencia=True,
               pitch=0, index_rate=0.6, protect=0.33, envolvente=1.0, limpiar=False,
               formato='mp3', normalizar=True, lufs=-16.0)
REFERENCIA = None   # la fija la celda 6
PRONUNCIACION = dict(PRONUNCIACION_DEFECTO)

def convertir_rvc(entrada, salida):
    args = ['--input-folder', entrada, '--output-folder', salida, '--pth-path', PTH, '--index-path', INDEX,
            '--pitch', AJUSTES['pitch'], '--index-rate', AJUSTES['index_rate'], '--protect', AJUSTES['protect'],
            '--volume-envelope', AJUSTES['envolvente'], '--f0-method', 'rmvpe', '--export-format', 'WAV']
    if AJUSTES['limpiar']:
        args += ['--clean-audio', '--clean-strength', 0.5]
    os.makedirs(salida, exist_ok=True)
    t = applio('batch-infer', *args, mostrar=False)
    for f in glob.glob(f'{salida}/*_output.wav'):  # Applio agrega "_output" al nombre
        os.replace(f, f[:-len('_output.wav')] + '.wav')
    if len(glob.glob(f'{salida}/*.wav')) < len(glob.glob(f'{entrada}/*.wav')):
        fallo('batch-infer', t, mostrado=False)

CLAVES_CLON = ('exageracion', 'cfg', 'temperatura', 'semilla', 'intentos', 'tomas_min', 'silabas_s',
               'verificar', 'asr', 'naturalidad', 'nitidez', 'fuerza_realce', 'limpiar_ruido',
               'silenciar_pausas', 'limpiar_referencia')
ULTIMO_INFORME = []

def clonar(trabajos):
    global ULTIMO_INFORME
    if AJUSTES['verificar']:
        print(f"  (por frase: {AJUSTES['tomas_min']}+ tomas → Whisper verifica que se entienda completa, "
              "UTMOS elige la más natural y Resemble Enhance la deja nítida a 44,1 kHz; "
              "la 1ª vez descarga los modelos)")
    ULTIMO_INFORME = remoto('clonar_lote', trabajos, REFERENCIA, {k: AJUSTES[k] for k in CLAVES_CLON}, mostrar=True)
    return ULTIMO_INFORME

def producir(pares, lote='estudio', mostrar=12, descargar=True, textos=True):
    if not pares:
        print('No hay textos para generar.'); return {}
    assert REFERENCIA, 'Falta la referencia de la voz: ejecuta la celda 6.'
    print(f'Generando {len(pares)} audio(s) con la voz clonada de la locutora '
          f'(expresividad {AJUSTES["exageracion"]}, ritmo {AJUSTES["cfg"]}, semilla {AJUSTES["semilla"]})…')
    salidas = L.generar(pares, None, f'/content/trabajo_{lote}',
                        convertir_rvc if (REFORZAR_CON_RVC and PTH) else None,
                        sintetizar_lote=clonar, max_car=250, pronunciacion=PRONUNCIACION,
                        formato=AJUSTES['formato'], normalizar=AJUSTES['normalizar'], lufs=AJUSTES['lufs'])
    texto_de = dict(pares)
    dudosos = sorted({pares[int(r['ruta'][:3]) - 1][0] for r in ULTIMO_INFORME if not r['ok']})
    if dudosos:
        print(f"\n⚠ {len(dudosos)} audio(s) para escuchar con atención: {', '.join(dudosos)}\n"
              f"  Si alguno no te convence: cambia la SEMILLA y regenera solo esos (celda 7: SOLO_ESTOS).")
    elif any(r['parecido'] is not None for r in ULTIMO_INFORME):
        print('\n✔ Todas las frases se entendieron completas (verificadas con Whisper).')
    else:
        print('\n(sin verificación Whisper: solo se revisó la duración de cada frase)')
    for i, (nombre, ruta) in enumerate(salidas.items()):
        if i == mostrar:
            print(f'… y {len(salidas) - mostrar} más en el ZIP.'); break
        detalle = f' — {texto_de[nombre][:90]}' if textos else ''
        display(HTML(f'<b>{nombre}</b> ({duracion(ruta):.1f} s){detalle}'))
        display(Audio(ruta))
    sello = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    if len(salidas) == 1:
        entrega = next(iter(salidas.values()))
    else:
        entrega = shutil.make_archive(f'/content/{lote}_{sello}', 'zip',
                                      root_dir=os.path.dirname(next(iter(salidas.values()))))
    if BACKUP:
        destino = f'{BACKUP}/salidas/{lote}_{sello}{os.path.splitext(entrega)[1]}'
        shutil.copy(entrega, destino); print('Copia en Drive:', destino)
    if descargar:
        from google.colab import files; files.download(entrega)
    return salidas

print('Motor listo ✔')

In [ ]:
#@title 6 · Elegir la mejor referencia de la locutora (automático; ~5 min la 1ª vez, se guarda en Drive)
RECALIBRAR = False  #@param {type:"boolean"}
#@markdown Prepara una referencia por cada `vof`, clona con cada una la misma frase del Modo Guiado y se queda con la que
#@markdown suena **más a la locutora** (verificador de hablante WavLM + tono, ritmo y entonación).
import glob, json, os, shutil, statistics

CLAVE = f'{HUELLA}:chatterbox'
CAL_PATH = f'{BACKUP}/modelo/voz_clonada.json' if BACKUP else '/content/voz_clonada.json'
REF_GUARDADA = f'{BACKUP}/modelo/referencia.wav' if BACKUP else '/content/referencia.wav'
CAL = None
if os.path.exists(CAL_PATH) and os.path.exists(REF_GUARDADA) and not RECALIBRAR:
    previo = json.load(open(CAL_PATH))
    CAL = previo if previo.get('clave') == CLAVE else None

if CAL is None:
    DATA_WAVS = sorted(glob.glob(f'{DATASET}/*.wav'))
    print('Analizando la voz de la locutora…')
    REF_RASGOS = analizar_varias([DATA_WAVS])[0]
    lufs_ref = statistics.median(medir_lufs(f) for f in sorted(glob.glob(f'{RAW}/*')))
    print(f"  tono {REF_RASGOS['f0']:.0f} Hz · ritmo {REF_RASGOS['silabas_s']:.1f} sílabas/s · "
          f"entonación {REF_RASGOS['rango_st']:.1f} st · volumen {lufs_ref:.1f} LUFS")
    CDIR = '/content/calibracion'
    shutil.rmtree(CDIR, ignore_errors=True)
    refs = remoto('preparar_referencias', DATA_WAVS, f'{CDIR}/refs')
    FRASE_CAL = GUION['guiado_regional'] + ' ' + GUION['guiado_dia']
    ajustes = {k: AJUSTES[k] for k in ('exageracion', 'cfg', 'temperatura', 'semilla', 'intentos')}
    ajustes['silabas_s'] = REF_RASGOS['silabas_s']
    print(f'Clonando la voz con {len(refs)} referencias (la 1ª vez descarga el modelo, ~2 GB)…')
    cands = remoto('clonar_candidatas', FRASE_CAL, [r['ruta'] for r in refs], f'{CDIR}/cand', ajustes)
    rasgos = analizar_varias(cands)
    print('Midiendo el parecido con el verificador de hablante (WavLM)…')
    sim = similitudes(DATA_WAVS, cands)
    if sim['sims'] is None:
        print(f"  (verificador no disponible: {sim['aviso']} → se decide por tono, ritmo y entonación)")
    filas = []
    for n, (ref, cand, ras) in enumerate(zip(refs, cands, rasgos)):
        dist = distancia_rasgos(REF_RASGOS, ras)
        s_ = sim['sims'][n] if sim['sims'] else None
        stem = os.path.splitext(os.path.basename(DATA_WAVS[n]))[0]
        clip = f'{stem} · {NOMBRES_VOF[stem]}' if stem in NOMBRES_VOF else stem
        filas.append(dict(referencia=ref['ruta'], clip=clip, segundos=ref['seg_propios'], candidato=cand,
                          similitud=s_, distancia=round(dist, 3),
                          puntaje=(s_ - 0.02 * dist) if s_ is not None else -dist))
    filas.sort(key=lambda f: -f['puntaje'])
    shutil.copy(filas[0]['referencia'], REF_GUARDADA)
    CAL = dict(clave=CLAVE, clip=filas[0]['clip'], lufs=lufs_ref, locutora=REF_RASGOS,
               ranking=[{k: v for k, v in f.items() if k not in ('referencia', 'candidato')} for f in filas])
    json.dump(CAL, open(CAL_PATH, 'w'), ensure_ascii=False, indent=1)
    print('\nRanking de referencias (mayor similitud y menor distancia = más parecida):')
    for n, f in enumerate(filas, 1):
        s_ = f"{f['similitud']:.3f}" if f['similitud'] is not None else '  —  '
        print(f"  {n}. {f['clip']} ({f['segundos']:.0f} s propios) · similitud {s_} · distancia {f['distancia']:.2f}")
    display(HTML('<b>Locutora original (vof):</b>')); display(Audio(max(DATA_WAVS, key=os.path.getsize)))
    for f in filas[:3]:
        display(HTML(f"<b>Voz clonada con referencia {f['clip']}</b>")); display(Audio(f['candidato']))

REFERENCIA = REF_GUARDADA
AJUSTES.update(silabas_s=CAL['locutora']['silabas_s'], lufs=CAL['lufs'])
print(f"\nReferencia elegida ✔ {CAL['clip']} (guardada en {REF_GUARDADA})")

In [ ]:
#@title 7 · Todas las voces de la app (Modo Guiado + tutoriales) → ZIP para `assets/vof_tutorial/`
GENERAR_VOCES_APP = True  #@param {type:"boolean"}
INCLUIR_TUTORIAL = True   #@param {type:"boolean"}
SOLO_ESTOS = ""           #@param {type:"string"}
#@markdown Para rehacer solo algunos: sus nombres separados por coma (p. ej. `guiado_hora, ficha_03`). Vacío = todos.
SEMILLA_APP = 1234        #@param {type:"integer"}
#@markdown Otra semilla = otra toma de la misma voz (úsala junto con `SOLO_ESTOS`).
#@markdown Genera **todas las voces de la app**: las 8 del Modo Guiado y, con `INCLUIR_TUTORIAL`, las 19 de los tutoriales (invitación, ficha, calendario y trámites).
#@markdown Extrae los mp3 del ZIP **directo** en `assets/vof_tutorial/` (sin subcarpeta).
if GENERAR_VOCES_APP:
    elegidos = {x.strip() for x in SOLO_ESTOS.split(',') if x.strip()}
    faltan = elegidos - set(GUION)
    assert not faltan, f'No existen en el guion: {sorted(faltan)}. Nombres válidos: {sorted(GUION)}'
    if elegidos:
        pares = [(k, v) for k, v in GUION.items() if k in elegidos]
    else:
        pares = [(k, v) for k, v in GUION.items() if INCLUIR_TUTORIAL or k.startswith('guiado_')]
    pares.sort(key=lambda p: not p[0].startswith('guiado_'))  # las del Modo Guiado primero (se escuchan arriba)
    previo = dict(AJUSTES)
    AJUSTES.update(formato='mp3', semilla=SEMILLA_APP)  # la app usa <id>.mp3
    try:
        producir(pares, lote='vof_tutorial', mostrar=8 if not elegidos else len(pares))
    finally:
        AJUSTES.update(formato=previo['formato'], semilla=previo['semilla'])
else:
    print('Omitido (marca GENERAR_VOCES_APP).')

## ✍️ Estudio — cualquier texto

Escribe en `TEXTOS` (celda 8) **una línea por audio**:

```
bienvenida | Bienvenido al sistema de citas de COSSMIL.
aviso_horario | Recuerde presentarse quince minutos antes. [pausa] Gracias por su preferencia.
Una línea sin nombre también sirve: se llamará clip_01_una-linea-sin-nombre.
# Las líneas con # se ignoran.
```

- `nombre | texto` → el archivo se llama `nombre.mp3` (útil para la app: `guiado_intro | …`).
- `[pausa]` = silencio de 0,6 s · `[pausa 1.5]` = 1,5 s · `[pausa 300ms]`.
- Los números se leen solos (`8:30` → "ocho y treinta", `21 fichas` → "veintiún fichas").
- Textos largos se parten solos en frases.
- Si una palabra suena mal, agrégala a `PRONUNCIACION` (p. ej. `'COSSMIL': 'Cossmil'`).

**Calidad:** por frase se generan al menos `TOMAS_MIN` tomas; Whisper descarta las cortadas o con balbuceo, UTMOS elige la más natural y Resemble Enhance la deja limpia y nítida (44,1 kHz). Al final verás cuáles conviene revisar.

**Ajustes:** `EXPRESIVIDAD` más alta = más emoción (0.5 es natural y sereno); `RITMO` más bajo = más pausado y articulado; `VARIACION` más baja = más estable; **`SEMILLA`**: otra toma distinta de la misma voz (si una frase no te gusta, cámbiala).

In [ ]:
#@title 8 · Estudio — escribe tus textos y ejecuta esta celda
TEXTOS = """
prueba_bienvenida | Bienvenido a COSSMIL. [pausa] Le acompañaré paso a paso para reservar su cita médica.
Este es un texto de prueba: puede escribir aquí cualquier cosa que necesite, y se generará con la voz de la locutora.
"""
REFERENCIA_VOZ = "automática (la mejor)"  #@param ["automática (la mejor)", "vof_01 · AUDIO 1. BIENVENIDA", "vof_02 · AUDIO 2. LOGIN", "vof_03 · AUDIO 3. ADVERTENCIA DE SEGURIDAD", "vof_04 · AUDIO 4. HORARIOS HABILITADOS CON HORA", "vof_05 · AUDIO 4.1. HORARIO HABILITADO SIN HORA", "vof_06 · AUDIO 5. FINAL CITA MEDICA REGISTRADA", "vof_07 · AUDIO 7. NOTIFICACION CITA MEDICA"]
EXPRESIVIDAD = 0.5   #@param {type:"slider", min:0.25, max:1.0, step:0.05}
RITMO = 0.4          #@param {type:"slider", min:0.2, max:0.8, step:0.05}
VARIACION = 0.75     #@param {type:"slider", min:0.4, max:1.2, step:0.05}
SEMILLA = 1234       #@param {type:"integer"}
NITIDEZ_ESTUDIO = True  #@param {type:"boolean"}
#@markdown Limpia con red neuronal y deja la voz a 44,1 kHz (Resemble Enhance). `FUERZA_LIMPIEZA` 0.9 = limpieza máxima (configuración oficial); bájala solo si notas la voz apagada.
FUERZA_LIMPIEZA = 0.9   #@param {type:"slider", min:0, max:1, step:0.05}
SILENCIAR_PAUSAS = True #@param {type:"boolean"}
#@markdown Baja el soplido de fondo en los huecos entre palabras (la voz no se toca).
ELEGIR_LA_MAS_NATURAL = True  #@param {type:"boolean"}
#@markdown Genera al menos `TOMAS_MIN` tomas por frase y se queda con la más humana (medidor UTMOS).
TOMAS_MIN = 3        #@param {type:"slider", min:1, max:6, step:1}
VERIFICAR_CON_WHISPER = True  #@param {type:"boolean"}
#@markdown Escucha cada frase y la rehace (hasta `TOMAS_MAX` veces) si sale cortada, con balbuceo o ruido.
TOMAS_MAX = 5        #@param {type:"slider", min:1, max:8, step:1}
FORMATO = "mp3"      #@param ["mp3", "wav"]
NORMALIZAR_VOLUMEN = True  #@param {type:"boolean"}
DESCARGAR = True     #@param {type:"boolean"}
NOMBRE_LOTE = "estudio"  #@param {type:"string"}
PRONUNCIACION.update({
    # 'palabra como se escribe': 'como debe sonar',
})

if REFERENCIA_VOZ.startswith('automática'):
    REFERENCIA = REF_GUARDADA
else:  # una vof concreta como referencia (preparada en la celda 6)
    REFERENCIA = f'/content/calibracion/refs/ref_{REFERENCIA_VOZ.split()[0]}.wav'
    if not os.path.exists(REFERENCIA):
        remoto('preparar_referencias', sorted(glob.glob(f'{DATASET}/*.wav')), '/content/calibracion/refs')
AJUSTES.update(exageracion=EXPRESIVIDAD, cfg=RITMO, temperatura=VARIACION, semilla=SEMILLA,
               nitidez=NITIDEZ_ESTUDIO, fuerza_realce=FUERZA_LIMPIEZA, naturalidad=ELEGIR_LA_MAS_NATURAL,
               silenciar_pausas=SILENCIAR_PAUSAS,
               tomas_min=TOMAS_MIN, verificar=VERIFICAR_CON_WHISPER, intentos=max(TOMAS_MAX, TOMAS_MIN),
               formato=FORMATO, normalizar=NORMALIZAR_VOLUMEN)
producir(parse_textos(TEXTOS), lote=nombre_seguro(NOMBRE_LOTE), descargar=DESCARGAR)

In [ ]:
#@title 9 · (Opcional) Generar desde un archivo .txt / .json / .csv
USAR_ARCHIVO = False  #@param {type:"boolean"}
#@markdown `.txt`: mismo formato que la celda 8 · `.json`: `{"id": "texto"}` o `[{"id":…, "texto":…}]` · `.csv`: columnas `id,texto`.
#@markdown Usa los ajustes de la celda 8.
if USAR_ARCHIVO:
    from google.colab import files
    pares = []
    for nombre, datos in files.upload().items():
        pares += parse_archivo(nombre, datos)
    producir(pares, lote='archivo', mostrar=6)
else:
    print('Omitido (marca USAR_ARCHIVO para subir un archivo de textos).')